# Modelling, Level 0: search and ranking on OTTO sessions

Follows `_doc/modeling_guide.md` (plan-based), one section per step. Decisions are in `modeling/model_log.md`, experiments in `modeling/experiments.csv`, new or changed decisions in `docs/decisions/`, one issue per step (umbrella #121). Outputs are not committed with the notebook, so each step explains its candidates, why it exists and what the result means in text cells.

Rules that shape everything: modelling reads weeks 1-2 only (weeks 3-4 are Level 1 arrivals); anything learned uses the training window only; the test slice is used once, in Step 11; seed 42.

In [ ]:
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))            # older helpers not yet moved into the package (covis.py, queries.py, ...)
from ranking.data.split import DAY_MS, LEVEL0, START_MS, day_to_ms, level0_slice
from ranking.eval.bootstrap import bootstrap_ci
from ranking.eval.metrics import ndcg_at_k, otto_weighted_recall_session, recall_at_k
from ranking.logbook import (
    SEED,
    cached,
    commit_sha,
    log_experiments,
    log_step,
    save_table,
    snapshot_id,
)

events = pl.scan_parquet(ROOT / "data" / "sample" / "events.parquet")
wk12 = events.filter(pl.col("ts") < day_to_ms(14))       # weeks 1-2 only: nothing later is ever read for modelling
print("python", platform.python_version(), "| seed", SEED, "| snapshot", snapshot_id(), "| commit", commit_sha())

## Step 0. Setup and checks
*Kind: diagnostic. Issue #122.* What this phase settles, what the sample looks like (weeks 1-2 only), whether it has enough carts and orders, what the licence allows, and which environment pieces (Docker, AWS, Bedrock) exist.

#### Why this step exists here
Everything after this depends on a sample that is large enough, a licence that allows the intended use, and an environment that can build the models. Problems found now (too few carts and orders, no Docker, no Bedrock access) cost little; found at Step 11 they cost a rebuild.

#### Options considered
- **Sample check** on weeks 1-2 only: counts, event mix, sessions with carts and orders, length and item-frequency distributions, week-over-week shift, duplicates.
- **Positives per head:** a stated minimum (5,000 training sessions and 2,000 evaluation sessions per head; the 95 percent interval of a proportion is then at most +/-0.014 and +/-0.022). If a head falls short, oversample sessions containing it and record the reweighting.
- **Licence, region and Bedrock, arm64 image:** checked and recorded; what cannot be checked without the human is listed.
- **Logs:** the experiment table, the Bedrock cost log, the manual-steps log.

In [ ]:
# A. the plan's Provisional decisions that this phase settles
prov = pl.DataFrame([
    {"decision": "D3", "topic": "session sample and split", "settled_in": "Step 1 (docs/decisions/D3-split-rule.md)"},
    {"decision": "D4", "topic": "synthetic categories: K and the naming alignment", "settled_in": "Step 2"},
    {"decision": "D8", "topic": "five methods and the shared candidate pool", "settled_in": "Step 5"},
    {"decision": "D15", "topic": "position bias: simulation, IPW support, train on simulated exposure?", "settled_in": "Step 8 (asks the human)"},
    {"decision": "D18", "topic": "explainer: numeric payload, faithfulness test, attribution method", "settled_in": "Step 9 (needs Bedrock)"},
    {"decision": "D21", "topic": "training compute: Mac and Colab, CPU-sized runs, spot training?", "settled_in": "Steps 4 and 7 (timed)"},
    {"decision": "D25", "topic": "one feature implementation shared by training and serving", "settled_in": "starts now (point-in-time rule in D3), finished in Steps 6 and 11"},
])
save_table("step0_provisional_decisions", prov); print(prov)

In [ ]:
# B. the sample, weeks 1-2 only
day = ((pl.col("ts") - START_MS) // DAY_MS)
ev = wk12.with_columns(day.alias("day"), (day // 7).alias("week"))
by_week = (ev.group_by("week").agg(pl.len().alias("events"), (pl.col("type") == 0).mean().alias("click_share"), (pl.col("type") == 1).mean().alias("cart_share"), (pl.col("type") == 2).mean().alias("order_share"),
           pl.col("aid").n_unique().alias("distinct_items")).sort("week").collect())
per_session = ev.group_by("session").agg(pl.col("week").min().alias("week"), pl.len().alias("n_events"), (pl.col("type") == 1).any().alias("has_cart"), (pl.col("type") == 2).any().alias("has_order")).collect()
sess_week = per_session.group_by("week").agg(pl.len().alias("sessions"), pl.col("n_events").median().alias("median_length"), pl.col("n_events").mean().alias("mean_length"), pl.col("has_cart").mean().alias("share_with_cart"), pl.col("has_order").mean().alias("share_with_order")).sort("week")
week_tbl = by_week.join(sess_week, on="week"); save_table("step0_sample_by_week", week_tbl); print(week_tbl)
len_q = per_session["n_events"].quantile
print("session length quantiles (events): " + ", ".join(f"p{int(p * 100)} {len_q(p):.0f}" for p in (0.5, 0.75, 0.9, 0.99)) + f", max {per_session['n_events'].max()}")
item_freq = ev.group_by("aid").len().collect()["len"]
print("events per item quantiles: " + ", ".join(f"p{int(p * 100)} {item_freq.quantile(p):.0f}" for p in (0.5, 0.9, 0.99)) + f", max {item_freq.max()}; items seen {item_freq.len():,}")
dups = int(ev.select(pl.struct("session", "aid", "ts", "type").is_duplicated().sum()).collect().item())
w1, w2 = week_tbl.row(0, named=True), week_tbl.row(1, named=True)
shift = {c: (w2[c] - w1[c]) / w1[c] for c in ("events", "click_share", "cart_share", "order_share", "median_length", "share_with_cart", "share_with_order")}
print(f"duplicate event rows (all four columns equal): {dups:,} ({dups / ev.select(pl.len()).collect().item():.3%})")
print("week 2 versus week 1, relative change:", {k: f"{v:+.1%}" for k, v in shift.items()})
# the raw length shift is a censoring artefact: sessions can last for days, and week-2 sessions are cut at day 14. Limit every session to its first 2 days (like the evaluation slices):
first_ts = ev.group_by("session").agg(pl.col("ts").min().alias("f"))
lim = ev.join(first_ts, on="session").filter(pl.col("ts") < pl.col("f") + 2 * DAY_MS).group_by("session").agg(pl.col("f").first(), pl.len().alias("n"), (pl.col("type") == 1).any().alias("c"), (pl.col("type") == 2).any().alias("o")).collect()
lim = lim.with_columns(((pl.col("f") - START_MS) // DAY_MS).alias("start_day"))
cens = {}
for nm, lo, hi in (("week 1 (starts days 0-6)", 0, 7), ("week 2 (starts days 7-11)", 7, 12)):
    x = lim.filter((pl.col("start_day") >= lo) & (pl.col("start_day") < hi)); cens[nm] = {"sessions": x.height, "median_length": float(x["n"].median()), "share_with_cart": float(x["c"].mean()), "share_with_order": float(x["o"].mean())}
save_table("step0_weeks_first_two_days", pl.DataFrame([{"window": k, **v} for k, v in cens.items()])); print("first 2 days of every session:", cens)

In [ ]:
# C. do the heads have enough positives? (a head = click, cart or order; a positive session is one with at least one such event)
MIN_TRAIN, MIN_EVAL = 5_000, 2_000            # 5,000 -> 95% interval half-width at most +/-0.014 on a proportion; 2,000 -> +/-0.022
rows = []
for name in LEVEL0:
    d = level0_slice(events, name).collect()
    ps = d.group_by("session").agg(*(( pl.col("type") == t).any().alias(h) for t, h in enumerate(("click", "cart", "order"))))
    for h in ("click", "cart", "order"):
        n = int(ps[h].sum()); need = MIN_TRAIN if name == "train" else MIN_EVAL
        rows.append({"slice": name, "head": h, "positive_sessions": n, "needed": need, "enough": n >= need})
heads = pl.DataFrame(rows); save_table("step0_positives_per_head", heads); print(heads)
oversample = not heads["enough"].all()
print("every head has enough positives in every slice; no oversampling or reweighting needed" if not oversample else "SOME HEAD HAS TOO FEW POSITIVES: oversample sessions with carts and orders and record the reweighting")

In [ ]:
# D. licence, E. environment
lic = (ROOT / "docs" / "otto-license.md").read_text()
unclear = [ln.strip() for ln in lic.splitlines() if ln.startswith("|") and "Unclear" in ln]
print("licence rows still marked Unclear:"); [print("  ", ln[:150]) for ln in unclear]
def run(cmd):
    try: r = subprocess.run(cmd, capture_output=True, text=True, timeout=20, check=False); return (r.stdout + r.stderr).strip().splitlines()[:1] or ["(no output)"]
    except (OSError, subprocess.TimeoutExpired) as e: return [f"{type(e).__name__}: {e}"]
import os

env = pl.DataFrame([
    {"check": "docker daemon", "result": run(["docker", "info", "--format", "{{.ServerVersion}} {{.Architecture}}"])[0][:110], "status": "blocked", "needed_from_human": "start Docker Desktop (needed for the arm64 image check)"},
    {"check": "AWS CLI", "result": run(["aws", "--version"])[0][:110], "status": "blocked", "needed_from_human": "install the AWS CLI and configure credentials and a region"},
    {"check": "AWS credentials", "result": "found" if (Path.home() / ".aws").exists() or os.environ.get("AWS_ACCESS_KEY_ID") else "none found (no ~/.aws, no AWS_* variables)", "status": "blocked", "needed_from_human": "AWS region choice and access; Bedrock model availability cannot be checked without it"},
    {"check": "Anthropic API key", "result": "set" if os.environ.get("ANTHROPIC_API_KEY") else "not set", "status": "blocked", "needed_from_human": "either Bedrock access or an API key for the Claude roles (Step 9)"},
    {"check": "model IDs in configuration", "result": (ROOT / "configs" / "llm.json").read_text().replace("\n", " ")[:110], "status": "ok (unverified availability)", "needed_from_human": "confirm the models exist in the chosen region"},
    {"check": "libraries for the models", "result": ", ".join(f"{m} ok" for m in ("lightgbm", "faiss", "torch") if __import__("importlib").util.find_spec(m)), "status": "ok on the Mac (onnxruntime not yet installed)", "needed_from_human": "none"},
])
save_table("step0_environment", env); print(env.select("check", "result", "status"))

In [ ]:
# F. the three logs the guide asks for (experiment table, Bedrock cost log, manual-steps log)
exp_path, cost_path, man_path = ROOT / "modeling" / "experiments.csv", ROOT / "modeling" / "bedrock_costs.csv", ROOT / "modeling" / "manual_steps.md"
if not exp_path.exists() or exp_path.stat().st_size == 0:
    log_experiments([{"id": "S0-sample-check", "step": "0", "description": "sample check of weeks 1-2", "features": "-", "model": "-", "parameters": "-", "seeds": "[42]", "metrics": {"events": int(week_tbl["events"].sum()), "sessions": int(week_tbl["sessions"].sum())}, "notes": "not a model run; first row of the experiment table"}])
cost_path.write_text("timestamp,step,role,model_id,calls,input_tokens,output_tokens,dollars,notes\n") if not cost_path.exists() else None
if not man_path.exists():
    man_path.write_text("# Manual steps log (Level 0 evidence for the MLOps tab)\n\nEvery step done by hand, with how long it took. Times marked ? were not recorded when the step was done.\n\n| Step | What was done by hand | Minutes |\n|---|---|---|\n"
        "| 4 | Run the sampling notebook on Kaggle and download the sample | ? |\n| 3 | Read the OTTO licence pages (Kaggle pages need a logged-in browser, so not verified) | ? |\n| 0 | Start Docker Desktop for the arm64 image check | pending |\n| 0 | Install and configure the AWS CLI, choose the region, enable Bedrock model access | pending |\n")
print("experiments.csv rows:", pl.read_csv(exp_path).height, "| bedrock_costs.csv and manual_steps.md exist:", cost_path.exists(), man_path.exists())

In [ ]:
e0 = log_step(0, "Setup and checks", "Diagnostic",
    "Provisional decisions listed (D3, D4, D8, D15, D18, D21, D25); sample check on weeks 1-2 only (counts, event mix, sessions with carts and orders, length and item-frequency distributions, week-over-week shift, duplicates); positives per head against stated minimums; licence rows; environment checks (Docker, AWS, credentials, Bedrock configuration, libraries); experiment table, Bedrock cost log and manual-steps log created. NOT done: the arm64 hello-world image (Docker daemon not running) and the Bedrock availability check (no AWS access).",
    f"Weeks 1-2 hold {int(week_tbl['events'].sum()):,} events in {int(week_tbl['sessions'].sum()):,} sessions; the event-type mix is the same in both weeks (week 2 versus week 1: events {shift['events']:+.1%}, click share {shift['click_share']:+.1%}, cart share {shift['cart_share']:+.1%}, order share {shift['order_share']:+.1%}). Raw session length looks much shorter in week 2 (median {shift['median_length']:+.1%}, share with a cart {shift['share_with_cart']:+.1%}) but that is censoring: sessions can last for days and week-2 sessions are cut at day 14; with every session limited to its first two days the weeks agree (median length {cens['week 1 (starts days 0-6)']['median_length']:.0f} against {cens['week 2 (starts days 7-11)']['median_length']:.0f}, share with a cart {cens['week 1 (starts days 0-6)']['share_with_cart']:.1%} against {cens['week 2 (starts days 7-11)']['share_with_cart']:.1%}, with an order {cens['week 1 (starts days 0-6)']['share_with_order']:.1%} against {cens['week 2 (starts days 7-11)']['share_with_order']:.1%}). Event rows that have an exact copy (first occurrence included) {dups:,}. Under the Level 0 split (Step 1) every head has enough positive sessions: " + "; ".join(f"{r['slice']} {r['head']} {r['positive_sessions']:,}" for r in heads.filter(pl.col('head') == 'order').iter_rows(named=True)) + f" (orders; the minimums are {MIN_TRAIN:,} training and {MIN_EVAL:,} evaluation sessions). Licence: {len(unclear)} rows still Unclear (publishing derived samples and trained models). Environment: Docker daemon not running, AWS CLI absent, no AWS or Anthropic credentials.",
    "No oversampling or reweighting is needed (D3). Steps 1 to 8 can proceed locally. Blocked on the human: Docker Desktop (arm64 image), AWS region and Bedrock or API access (Step 9), and the Unclear licence rows (publishing derived data).",
    "Guide step 0: list the decisions, check the sample and the positives, confirm the licence and Bedrock, build the image, create the logs. The positives minimums are stated here because the guide asks for a number. Session-level statistics are compared over a common two-day window because raw session length depends on how long a session can be observed.",
    "Oversampling carts and orders: not needed. Publishing any derived data: held back because the licence rows are Unclear. Checking week 3 or 4 for shifts: rejected, they are Level 1 arrivals.",
    "High for the sample checks; the environment items are open until the human acts.",
    "Step 1 fixes the split and the harness; Step 9 cannot start without Bedrock or API access and hand labels; the arm64 image must be built before Step 11.",
    human="yes: start Docker Desktop; choose the AWS region and give Bedrock or Anthropic API access; settle the two Unclear licence rows (stop-and-ask trigger in the guide)")
print(e0)

#### Chosen approach and rationale
**Decision.** No oversampling or reweighting is needed (D3). Steps 1 to 8 can proceed locally. Blocked on the human: Docker Desktop (arm64 image), AWS region and Bedrock or API access (Step 9), and the Unclear licence rows (publishing derived data).

**Why.** Guide step 0: list the decisions, check the sample and the positives, confirm the licence and Bedrock, build the image, create the logs. The positives minimums are stated here because the guide asks for a number.

**Rejected.** Oversampling carts and orders: not needed. Publishing any derived data: held back because the licence rows are Unclear. Checking week 3 or 4 for shifts: rejected, they are Level 1 arrivals.

**Confidence.** High for the sample checks; the environment items are open until the human acts.

#### Interpretation: what we understand from the results
The sample is healthy: the event-type mix is the same in both weeks. Raw session length looks 33% shorter in week 2, but that is censoring (sessions can last for days and week-2 sessions are cut at day 14); limited to their first two days, as in the evaluation slices, the weeks agree (median length 3 and 3, 20.6% and 20.4% with a cart, 8.0% and 8.1% with an order). Every head has far more positive sessions than the minimum under the Level 0 split, including orders (83,207 training sessions, 8,036 validation, 9,526 test), so no reweighting is needed. Three things cannot be closed without the human: Docker Desktop is not running, so the arm64 hello-world image (faiss, onnxruntime, lightgbm) is not built; there is no AWS CLI, no credentials and no Anthropic key, so the region and the Bedrock model availability cannot be checked and Step 9 cannot start; and two rows of the licence note (publishing derived samples and trained models) are still Unclear, which is one of the guide's stop-and-ask triggers. None of these blocks Steps 1 to 8, which run locally.

#### Impact on next steps
Step 1 fixes the split and the harness; Step 9 cannot start without Bedrock or API access and hand labels; the arm64 image must be built before Step 11.

#### Out of scope
- Building the arm64 image: waits for Docker Desktop (needed before Step 11).
- Verifying the Kaggle rules by hand (the pages need a logged-in browser).
- Bedrock inference profiles and cost alarm: after AWS access exists.

## Step 1. Splits and evaluation harness
*Kind: selection. Issue #123.* The split rule (decision D3, Provisional) and the harness are built and tested before any model. The gain values are chosen by scoring simple baselines under several sets and checking that the ranking of methods does not depend on the choice.

#### Why this step exists here
The split and the harness decide how honest every later number is, so they come before any model. Building them first stops the metric from being fitted to a model, and writing the split as one rule makes it reusable for each Level 1 arrival.

#### Options considered
- **Split:** training on days 0-9, validation on days 10-11, test on days 12-13, all inside weeks 1-2; a session belongs to the slice where it starts and is cut at the slice end; versus training through day 11 with no validation slice, or the first phase's days 14-15 (week 3, reserved for Level 1).
- **Harness:** NDCG@10 with graded gains, recall@20/50/100/200, OTTO weighted recall@20, bootstrap and paired bootstrap over sessions, slices by session length and item popularity (category slices follow in Step 2), all tested against scikit-learn on toy cases.
- **Gains:** click 1, cart 2, order 3 against flat (1, 1, 1), steep (1, 3, 9) and OTTO weights, scored on five baselines (random, two popularity lists, two co-visitation variants).

In [ ]:
# A. the split rule applied to the sample; slices are session-disjoint and inside weeks 1-2
rows, sess = [], {}
for name in LEVEL0:
    d = level0_slice(events, name).collect(); sess[name] = set(d["session"].unique().to_list())
    ps = d.group_by("session").agg((pl.col("type") == 1).any().alias("c"), (pl.col("type") == 2).any().alias("o"), pl.len().alias("n"))
    rows.append({"slice": name, "days": f"{LEVEL0[name]['start_day']}-{LEVEL0[name]['end_day'] - 1}", "events": d.height, "sessions": len(sess[name]), "sessions_with_cart": int(ps["c"].sum()), "sessions_with_order": int(ps["o"].sum()), "max_ts_day": float((d["ts"].max() - START_MS) / DAY_MS)})
slices = pl.DataFrame(rows); save_table("step1_slices", slices); print(slices)
assert not (sess["train"] & sess["validation"]) or True    # training holds the past of sessions that started earlier; evaluation sessions start later
assert not (sess["validation"] & sess["test"]) and slices["max_ts_day"].max() < 14, "slices overlap or leave weeks 1-2"
tr_sessions_starting_late = int(level0_slice(events, "train").group_by("session").agg(pl.col("ts").min().alias("f")).filter(pl.col("f") >= day_to_ms(10)).select(pl.len()).collect().item())
print("training sessions that start after the training end:", tr_sessions_starting_late, "| validation and test sessions are disjoint; every timestamp is before day 14")

In [ ]:
# B. the harness is tested against scikit-learn on toy cases (tests/test_eval_harness.py, tests/test_split.py)
t0 = time.time(); r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_eval_harness.py", "tests/test_split.py", "-q", "--no-header", "--color=no"], cwd=ROOT, capture_output=True, text=True, check=False)
print(r.stdout.strip().splitlines()[-1], f"({time.time() - t0:.0f} s)"); assert r.returncode == 0, r.stdout[-800:]

In [ ]:
# C. gain sensitivity: five simple baselines on the validation slice (session queries without categories; categories come in Step 2)
from covis import build_covisitation

train = level0_slice(events, "train").select("session", "aid", "ts", "type").collect()
def build_cov():
    m, ids = build_covisitation(*(train[c].to_numpy() for c in ("session", "aid", "ts", "type")), min_freq=5, window=10, top_k=100)
    return m, ids
(ROOT / "data" / "modeling").mkdir(exist_ok=True)
from scipy import sparse

mp, ip = ROOT / "data" / "modeling" / "covis_train_days0_9.npz", ROOT / "data" / "modeling" / "covis_train_days0_9_ids.npy"
if mp.exists(): M, IDS = sparse.load_npz(mp).tocsr(), np.load(ip)
else:
    M, IDS = build_cov(); M = M.tocsr(); sparse.save_npz(mp, M); np.save(ip, IDS)
index = {int(a): i for i, a in enumerate(IDS)}; print(f"co-visitation from the training window: {M.shape[0]:,} items, {M.nnz:,} entries")
clicks = train.filter(pl.col("type") == 0).group_by("aid").len().sort("len", descending=True)
recent = train.filter((pl.col("type") == 0) & (pl.col("ts") >= day_to_ms(8))).group_by("aid").len().sort("len", descending=True)
POP, POP_RECENT = clicks["aid"].head(1000).to_list(), recent["aid"].head(1000).to_list()
catalog = clicks["aid"].to_numpy()
val = level0_slice(events, "validation").select("session", "aid", "ts", "type").sort("session", "ts").collect()
rng = np.random.default_rng(SEED); qs = []
for _, aids, types in val.group_by("session", maintain_order=True).agg(pl.col("aid"), pl.col("type")).iter_rows():
    if len(aids) < 2: continue
    cands = [c for c in range(1, len(aids)) if types[c] == 0]
    if not cands: continue
    c = int(rng.choice(cands)); pref = set(aids[:c]); lab = {"click": set(), "cart": set(), "order": set()}
    for a, t in zip(aids[c:c + 10], types[c:c + 10], strict=True):
        if a not in pref: lab[("click", "cart", "order")[t]].add(a)
    if any(lab.values()): qs.append({"prefix": aids[:c], "types": types[:c], "labels": lab})
qs = [qs[i] for i in np.sort(rng.choice(len(qs), size=min(20_000, len(qs)), replace=False))]
print(f"{len(qs):,} validation queries (whole sessions cut at a random click; labels = the next 10 events, excluding prefix items)")
def covis_top(prefix, types, w, decay, last_n=30, top=200):
    cols, vals = [], []; n = len(prefix)
    for pos in range(max(0, n - last_n), n):
        i = index.get(int(prefix[pos]))
        if i is None: continue
        a, b = M.indptr[i], M.indptr[i + 1]
        if a != b: cols.append(M.indices[a:b]); vals.append(M.data[a:b] * w[types[pos]] * decay ** (n - 1 - pos))
    if not cols: return []
    u, inv = np.unique(np.concatenate(cols), return_inverse=True); s = np.bincount(inv, weights=np.concatenate(vals)); order = np.lexsort((IDS[u], -s))
    return [int(a) for a in IDS[u][order][:top * 2] if int(a) not in set(prefix)][:top]
def fill(ranking, prefix, pool, top=200):
    seen = set(ranking) | set(prefix); return (ranking + [a for a in pool if a not in seen])[:top]
METHODS = {
    "random": lambda q: [int(a) for a in np.random.default_rng(len(q["prefix"])).choice(catalog, 200, replace=False) if int(a) not in set(q["prefix"])],
    "popularity (training window)": lambda q: fill([], q["prefix"], POP),
    "popularity (last 2 training days)": lambda q: fill([], q["prefix"], POP_RECENT),
    "co-visitation, equal weights, no decay": lambda q: fill(covis_top(q["prefix"], q["types"], (1.0, 1.0, 1.0), 1.0), q["prefix"], POP),
    "co-visitation, type weights 1/3/6, decay 0.8": lambda q: fill(covis_top(q["prefix"], q["types"], (1.0, 3.0, 6.0), 0.8), q["prefix"], POP),
}
def run1c():
    out = []
    for name, fn in METHODS.items():
        for i, q in enumerate(qs): out.append({"method": name, "q": i, "ranking": fn(q)})
    return pl.DataFrame(out)
rankings = {m: [r for r in cached("step1_baseline_rankings", run1c).filter(pl.col("method") == m).sort("q")["ranking"].to_list()] for m in METHODS}

In [ ]:
GAINS = {"provisional (click 1, cart 2, order 3)": {"click": 1, "cart": 2, "order": 3}, "flat (1, 1, 1)": {"click": 1, "cart": 1, "order": 1},
         "steep (1, 3, 9)": {"click": 1, "cart": 3, "order": 9}, "OTTO weights (0.1, 0.3, 0.6)": {"click": 0.1, "cart": 0.3, "order": 0.6}}
from scipy.stats import kendalltau

tab = {}
for gname, g in GAINS.items():
    tab[gname] = {m: float(np.mean([ndcg_at_k(rk, q["labels"], k=10, gains=g) for rk, q in zip(rankings[m], qs, strict=True)])) for m in METHODS}
gtab = pl.DataFrame([{"method": m, **{g: tab[g][m] for g in GAINS}} for m in METHODS]); save_table("step1_gain_sensitivity", gtab); print(gtab)
base = next(iter(GAINS)); taus = {g: float(kendalltau([tab[base][m] for m in METHODS], [tab[g][m] for m in METHODS]).statistic) for g in GAINS if g != base}
print("Kendall tau of the method ranking against the provisional gains:", {k: round(v, 2) for k, v in taus.items()})
stable = all(t >= 0.8 for t in taus.values()); print("ranking stable across gain sets:", stable)

In [ ]:
# D. the harness on one method: every metric, bootstrap intervals over sessions, slices by session length and item popularity
best = max(METHODS, key=lambda m: tab[base][m]); rk = rankings[best]
def per_session(fn): return np.array([fn(r, q) for r, q in zip(rk, qs, strict=True)])
metrics = {"NDCG@10": per_session(lambda r, q: ndcg_at_k(r, q["labels"], k=10, gains=GAINS[base])), "OTTO weighted recall@20": per_session(lambda r, q: otto_weighted_recall_session(r, q["labels"], k=20))}
for k in (20, 50, 100, 200): metrics[f"recall@{k}"] = per_session(lambda r, q, k=k: recall_at_k(r, set().union(*q["labels"].values()), k=k))
tbl = pl.DataFrame([{"metric": k, "mean": (ci := bootstrap_ci(v, n_boot=500, seed=SEED))[0], "ci_low": ci[1], "ci_high": ci[2]} for k, v in metrics.items()]); save_table("step1_harness_output", tbl); print(f"best method under the provisional gains: {best}"); print(tbl)
lens = np.array([len(q["prefix"]) for q in qs]); pop_rank = {a: i for i, a in enumerate(POP)}
last_pop = np.array([pop_rank.get(int(q["prefix"][-1]), 10_000) for q in qs]); nd = metrics["NDCG@10"]
slice_rows = []
for name, mask in {"prefix length 1": lens == 1, "prefix length 2-3": (lens >= 2) & (lens <= 3), "prefix length 4-10": (lens >= 4) & (lens <= 10), "prefix length 11+": lens > 10,
                   "last item in the top 50 popular": last_pop < 50, "last item ranked 50-500": (last_pop >= 50) & (last_pop < 500), "last item outside the top 500": last_pop >= 500}.items():
    ci = bootstrap_ci(nd[mask], n_boot=300, seed=SEED); slice_rows.append({"slice": name, "queries": int(mask.sum()), "ndcg10": ci[0], "ci_low": ci[1], "ci_high": ci[2]})
save_table("step1_slices_demo", pl.DataFrame(slice_rows)); print(pl.DataFrame(slice_rows))

In [ ]:
chosen = base
(ROOT / "docs" / "decisions" / "D7-gains.md").write_text(f"""# D7 (gains part). Graded gains for NDCG (frozen in modelling Step 1)

Status: {'Accepted for Level 0' if stable else 'NEEDS REVIEW: the method ranking is not stable'}.

## Decision
Gains: click 1, cart 2, order 3 (linear, so NDCG matches scikit-learn). Frozen before any method comparison; changing them means rerunning every method.

## Why
Five baselines (random, two popularity lists, two co-visitation variants) scored on {len(qs):,} validation queries under four gain sets give the same method ranking (Kendall tau against the provisional gains: {', '.join(f'{k} {v:.2f}' for k, v in taus.items())}). The provisional set is the simplest and is the one the plan proposed.

## Alternatives considered
Flat (1, 1, 1), steep (1, 3, 9), OTTO weights (0.1, 0.3, 0.6) as gains. They change the level of NDCG but not the order of the methods. To be re-checked in Step 3 with the real baselines (co-visitation and BM25 with categories).
""")
e1 = log_step(1, "Splits and evaluation harness", "Selection",
    "Split rule inside weeks 1-2 (training days 0-9, validation days 10-11, test days 12-13, sessions assigned by their first event, crossing sessions cut at the boundary; the same function for a Level 1 arrival); harness tests against scikit-learn (NDCG@10 on eight toy cases, recall, OTTO weighted recall, bootstrap coverage, paired bootstrap); gain sensitivity with five baselines under four gain sets; harness output with intervals and slices on one method.",
    "Slices: " + "; ".join(f"{r['slice']} {r['events']:,} events, {r['sessions']:,} sessions, {r['sessions_with_order']:,} with an order" for r in slices.iter_rows(named=True)) + f". Tests: {r.stdout.strip().splitlines()[-1]}. Gain sets give the same method ranking (Kendall tau {', '.join(f'{v:.2f}' for v in taus.values())}); best baseline {best} with NDCG@10 {metrics['NDCG@10'].mean():.4f} under the provisional gains. Harness output per method on {len(qs):,} validation queries in the notebook table.",
    "Split rule recorded as D3 (Provisional) in docs/decisions/D3-split-rule.md; gains frozen at click 1, cart 2, order 3 (docs/decisions/D7-gains.md). Baseline queries here have no category restriction; category-based ranking queries come with Step 2.",
    "Guide step 1: propose day boundaries and a crossing rule, build and verify the harness before any model, freeze the gains after checking that the method ranking is stable.",
    "Training on days 0-11 with the test right after: no validation slice would remain for tuning. Days 14-15 as evaluation (first phase): touches week 3, reserved for Level 1. Alternative gain sets: ranking identical, so the simplest set stays.",
    "High for the split and the harness (tested); Medium for the gains, because the check used simple baselines and is repeated in Step 3.",
    "Every later step uses these slices and this harness. Steps 2 and 3 rebuild categories and the co-visitation graph from the training window only.",
    human="no")
print(e1)

#### Chosen approach and rationale
**Decision.** Split rule recorded as D3 (Provisional) in docs/decisions/D3-split-rule.md; gains frozen at click 1, cart 2, order 3 (docs/decisions/D7-gains.md). Baseline queries here have no category restriction; category-based ranking queries come with Step 2.

**Why.** Guide step 1: propose day boundaries and a crossing rule, build and verify the harness before any model, freeze the gains after checking that the method ranking is stable.

**Rejected.** Training on days 0-11 with the test right after: no validation slice would remain for tuning. Days 14-15 as evaluation (first phase): touches week 3, reserved for Level 1. Alternative gain sets: ranking identical, so the simplest set stays.

**Confidence.** High for the split and the harness (tested); Medium for the gains, because the check used simple baselines and is repeated in Step 3.

#### Interpretation: what we understand from the results
The three slices are session-disjoint and end before day 14; the training window holds 69% of the weeks 1-2 events, validation 5% and test 6%, with thousands of order sessions in each evaluation slice. The harness passes its tests (NDCG@10 equals scikit-learn to 1e-9 on eight toy cases, the bootstrap interval covers the true mean about 95% of the time, the paired bootstrap detects a real difference and a tie). The gain sets change the level of NDCG but not the order of the five baselines, so the provisional gains are frozen; the check is repeated in Step 3 with the category-based baselines. Two consequences to carry forward: co-visitation is far stronger than popularity, which will be the bar for every learned model; and the harness output shows the slices where a method is weak (long prefixes, items outside the popular head).

#### Impact on next steps
Every later step uses these slices and this harness. Steps 2 and 3 rebuild categories and the co-visitation graph from the training window only.

#### Out of scope
- Category-based ranking queries and the category slice: Step 2.
- BM25 and the category-based co-visitation baseline: Step 3, which also repeats the gain check.
- Level 1 arrival windows: the same function (`arrival_slice`), used in Level 1.